# RiskForge — Phase 1: Data Ingestion, Memory Optimization & Statistical EDA
**Module Scope:** 1. Data Source | 2. Ingestion & Data Engineering | 3. Data Exploration & Analysis

This notebook demonstrates the end-to-end data ingestion, memory optimization, data cleaning, and statistical exploratory data analysis for the IEEE-CIS Fraud Detection dataset.

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so riskforge subpackages can be imported
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from riskforge.utils.config import get_project_root
from riskforge.utils.plotting import set_corporate_style, PALETTE, save_figure

set_corporate_style()
root = get_project_root()
print(f"[*] Project Root: {root}")

## 1. Load Cleaned Dataset
We load the processed dataset from `data/processed/clean_transactions.parquet`. Notice how fast Parquet loads compared to raw CSVs.

In [ ]:
parquet_path = root / "data" / "processed" / "clean_transactions.parquet"
df = pd.read_parquet(parquet_path)

print(f"[*] Total Transactions: {len(df):,}")
print(f"[*] Total Features: {df.shape[1]}")
print(f"[*] Memory Consumption: {df.memory_usage().sum() / 1024**2:.2f} MB")
df[['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'card4', 'card6', 'os_clean', 'browser_clean']].head(10)

## 2. Target Variable & Class Imbalance Analysis
In real financial payment networks, fraud is rare (<3.5%). Measuring standard model accuracy is flawed because a naive model predicting all zeros achieves 96.5% accuracy.

In [ ]:
counts = df['isFraud'].value_counts()
percentages = df['isFraud'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Transaction Count': counts,
    'Percentage (%)': percentages.round(2)
})
target_summary.index = ['Legitimate (0)', 'Fraudulent (1)']
display(target_summary)

## 3. Financial Risk Asymmetry (Count Volume vs. Dollar Loss)
Here we visualize the core business problem: how a 3.5% transaction count translates into outsized financial dollar loss.

In [ ]:
total_txns = len(df)
total_fraud_txns = df["isFraud"].sum()
fraud_txn_pct = (total_fraud_txns / total_txns) * 100
legit_txn_pct = 100 - fraud_txn_pct

total_dollars = df["TransactionAmt"].sum()
fraud_dollars = df.loc[df["isFraud"] == 1, "TransactionAmt"].sum()
fraud_dollar_pct = (fraud_dollars / total_dollars) * 100
legit_dollar_pct = 100 - fraud_dollar_pct

fig, ax = plt.subplots(figsize=(9, 5.5))
categories = ["Transaction Count Volume", "Gross Dollar Volume ($)"]
bars_legit = [legit_txn_pct, legit_dollar_pct]
bars_fraud = [fraud_txn_pct, fraud_dollar_pct]

bar_width = 0.45
x = np.arange(len(categories))

p1 = ax.bar(x, bars_legit, width=bar_width, label="Legitimate", color=PALETTE["legit"], edgecolor="white")
p2 = ax.bar(x, bars_fraud, width=bar_width, bottom=bars_legit, label="Fraudulent", color=PALETTE["fraud"], edgecolor="white")

ax.set_ylabel("Percentage of Total (%)")
ax.set_title("Financial Risk Asymmetry: Count vs. Dollar Loss Exposure", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 108)
ax.legend(loc="upper right", frameon=True)
plt.show()

## 4. Temporal Fraud Velocity & Attack Heatmap
Decomposing the chronological seconds timestamp (`TransactionDT`) into Hour of Day and Day of Week.

In [ ]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["day_of_week"] = (df["TransactionDT"] // 86400) % 7

heatmap_data = df.groupby(["day_of_week", "hour"])["isFraud"].mean().unstack() * 100
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heatmap_data.index = [day_labels[i] for i in heatmap_data.index]

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.heatmap(heatmap_data, cmap="YlOrRd", annot=False, fmt=".1f", linewidths=0.5, cbar_kws={'label': 'Empirical Fraud Rate (%)'}, ax=ax)
ax.set_title("Temporal Fraud Density: Empirical Fraud Rate by Hour & Day", pad=15)
ax.set_xlabel("Hour of Day (UTC)", fontsize=11)
ax.set_ylabel("Day of Week", fontsize=11)
plt.show()